# House Price Prediction
## Thiranex Internship — Task 3: Predictive Analytics Using Historical Data

**Objective:** Build a Linear Regression model to predict house prices based on structural and locational features.

**Dataset:** 500 synthetic records with 8 features  
**Algorithm:** Linear Regression  
**Evaluation:** R², MAE, RMSE


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('house_prices.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Basic Statistics ---")
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Feature Distributions', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flatten(), ['Area_sqft', 'Price_USD', 'Age_years', 'Crime_Rate']):
    ax.hist(df[col], bins=25, color='#2563EB', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df['Area_sqft'], df['Price_USD']/1e3, alpha=0.5, color='#2563EB', s=30)
axes[0].set_xlabel('Area (sqft)'); axes[0].set_ylabel('Price ($ thousands)')
axes[0].set_title('Area vs Price')

axes[1].scatter(df['Dist_City_km'], df['Price_USD']/1e3, alpha=0.5, color='#D97706', s=30)
axes[1].set_xlabel('Distance to City (km)'); axes[1].set_ylabel('Price ($ thousands)')
axes[1].set_title('City Distance vs Price')

plt.suptitle('Key Feature Relationships', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
X = df.drop('Price_USD', axis=1)
y = df['Price_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

## 5. Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("Model trained successfully!")
print(f"Intercept : ${model.intercept_:,.2f}")
print("\nCoefficients:")
for feat, coef in sorted(zip(X.columns, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:20s}: {coef:>12,.2f}")

## 6. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 40)
print("       MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"  R² Score : {r2:.4f}  ({r2*100:.1f}% variance explained)")
print(f"  MAE      : ${mae:>12,.2f}")
print(f"  RMSE     : ${rmse:>12,.2f}")
print("=" * 40)

## 7. Visualisations

In [ ]:
# Chart 1 — Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test/1e3, y_pred/1e3, alpha=0.55, color='#2563EB', edgecolors='white', s=60)
mn = min(y_test.min(), y_pred.min())/1e3
mx = max(y_test.max(), y_pred.max())/1e3
plt.plot([mn, mx], [mn, mx], '--', color='#DC2626', lw=1.8, label='Perfect fit')
plt.xlabel('Actual Price ($ thousands)'); plt.ylabel('Predicted Price ($ thousands)')
plt.title('Actual vs Predicted House Prices', fontsize=14, fontweight='bold')
plt.text(0.05, 0.92, f'R² = {r2:.3f}', transform=plt.gca().transAxes,
         fontsize=12, color='#16A34A', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Chart 2 — Feature Importance
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=True).index)
colors_bar = ['#16A34A' if c > 0 else '#DC2626' for c in coef_df['Coefficient']]

plt.figure(figsize=(8, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, edgecolor='white')
plt.axvline(0, color='grey', lw=0.8, linestyle='--')
plt.xlabel('Standardised Coefficient')
plt.title('Feature Importance (Linear Regression Coefficients)', fontsize=13, fontweight='bold')
pos_p = mpatches.Patch(color='#16A34A', label='Positive impact')
neg_p = mpatches.Patch(color='#DC2626', label='Negative impact')
plt.legend(handles=[pos_p, neg_p]); plt.tight_layout(); plt.show()

In [ ]:
# Chart 3 — Residual Analysis
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(residuals/1e3, bins=30, color='#2563EB', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='#DC2626', lw=1.8, linestyle='--')
axes[0].set_xlabel('Residual ($ thousands)'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Residuals Distribution', fontweight='bold')

axes[1].scatter(y_pred/1e3, residuals/1e3, alpha=0.5, color='#D97706', edgecolors='white', s=55)
axes[1].axhline(0, color='#DC2626', lw=1.8, linestyle='--')
axes[1].set_xlabel('Predicted Price ($ thousands)'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted', fontweight='bold')

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Summary

| Metric | Value |
|--------|-------|
| R² Score | 0.9825 |
| MAE | $14,046 |
| RMSE | $17,491 |

**Key Takeaways:**
- The model explains **98.2%** of the variance in house prices.
- Average prediction error is approximately **$14,046**.
- `Area_sqft` and `Dist_City_km` are the most influential features.
- Residuals are approximately normally distributed — the linear assumption holds well.

**Potential Improvements:**
- Try Random Forest or XGBoost for better accuracy.
- Add cross-validation (K-Fold).
- Incorporate more features (neighbourhood ratings, floor level, etc.).
